# 08.4 - Encoder-Decoder Architecture

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

Some tasks change structure: a source sequence of one length becomes a target sequence of another length (translation, summarization, reversing). The original transformer design pairs an **encoder** that reads the whole source with a **decoder** that writes the target one token at a time.

## 2. Why Does This Matter?

This is the "Attention Is All You Need" (2017) architecture. It is the ancestor of T5 and BART and still the right choice for any task with genuinely different input/output structures. Cross-attention and teacher forcing are two ideas you will use everywhere.

## 3. Prerequisites

- Transformer block (08.1), self-attention + causal mask (08.2), positional encoding (08.3)

## 4. Learning Objectives

By the end of this unit, you should:
- Connect an encoder stack to a decoder stack with cross-attention
- Use teacher forcing during training
- Decode greedily at inference
- Explain why the decoder needs a causal mask and the encoder does not

## 5. Mental Model

```text
Input -> Encoder stack -> memory
                              \
                               Decoder stack <- previously written tokens
                              /
                        Output token

Encoder: reads everything (bidirectional).
Decoder: writes one token at a time (causal).
Cross-attention (in decoder): Q from decoder, K & V from encoder memory.
```


## 6. Setup


In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print('torch', torch.__version__)


## 7. Build the Encoder-Decoder Transformer

We use PyTorch's encoder/decoder layers: the decoder layer internally contains masked self-attention, **cross-attention**, and a feed-forward network.


In [ ]:
class EncoderDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=32, n_heads=4, n_layers=2, d_ff=128, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        PE = self._sinusoidal(512, d_model)
        self.register_buffer('pe', PE)

        enc_layer = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, n_layers)

        dec_layer = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout=dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, n_layers)
        self.out = nn.Linear(d_model, vocab_size)

    @staticmethod
    def _sinusoidal(max_len, d):
        pos = torch.arange(max_len).unsqueeze(1).float()
        dv = torch.exp(torch.arange(0, d, 2).float() * -(np.log(10000.0) / d))
        pe = torch.zeros(max_len, d)
        pe[:, 0::2] = torch.sin(pos * dv)
        pe[:, 1::2] = torch.cos(pos * dv)
        return pe.unsqueeze(0)

    def forward(self, src, tgt, tgt_mask=None):
        se = self.emb(src) + self.pe[:, :src.size(1)]
        te = self.emb(tgt) + self.pe[:, :tgt.size(1)]
        memory = self.encoder(se)               # bidirectional self-attn over source
        dec_out = self.decoder(te, memory, tgt_mask=tgt_mask)  # cross-attn to memory
        return self.out(dec_out), memory

model = EncoderDecoder(vocab_size=20)
src = torch.randint(1, 20, (2, 8))    # batch=2, source length 8
tgt = torch.randint(1, 20, (2, 6))    # batch=2, target length 6
logits, memory = model(src, tgt)
print('src    :', tuple(src.shape))
print('memory :', tuple(memory.shape))
print('logits :', tuple(logits.shape), '-> vocab probs for every target position')


## 8. Cross-Attention: Decoder Looks at Encoder Memory

Prove the decoder actually consumes encoder memory: perturb the encoder output and watch decoder logits move.


In [ ]:
with torch.no_grad():
    logits_a, mem_a = model(src, tgt)
    # scramble encoder memory
    mem_b = torch.randn_like(mem_a)
    te = model.emb(tgt) + model.pe[:, :tgt.size(1)]
    dec_b = model.decoder(te, mem_b)
    logits_b = model.out(dec_b)
    diff = (logits_a - logits_b).abs().mean().item()
print(f'Mean |logits change| after scrambling encoder memory: {diff:.3f}')
print('If cross-attention were disconnected this would be ~0. It is not -'
      'every decoder token attends to the whole source.')


## 9. Task: Reverse a Sequence (a tiny seq2seq)

We train on '0 3 1 7' -> '7 1 3 0'. Teacher forcing: the input to the decoder is BOS + the *gold* previous tokens during training.


In [ ]:
V = 8        # tokens 1..7
L = 6        # sequence length
BOS, PAD = 0, 0

def gen_batch(batch, L=L, V=V):
    src = torch.randint(1, V + 1, (batch, L))
    tgt = torch.flip(src, dims=[1])
    dec_in = torch.cat([torch.full((batch, 1), BOS), tgt[:, :-1]], dim=1)
    return src, dec_in, tgt

def causal_mask(Lm):
    return torch.tril(torch.ones(Lm, Lm, dtype=torch.bool))

src, dec_in, tgt = gen_batch(4)
print('src   :', src[0].tolist())
print('dec_in:', dec_in[0].tolist())
print('tgt   :', tgt[0].tolist())


## 10. Train with Teacher Forcing


In [ ]:
model = EncoderDecoder(vocab_size=V + 1, d_model=32, n_heads=4, n_layers=2, d_ff=128, dropout=0.0)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD)

for step in range(1, 1501):
    src, dec_in, tgt = gen_batch(64)
    opt.zero_grad()
    logits, _ = model(src, dec_in, tgt_mask=causal_mask(L))
    loss = loss_fn(logits.reshape(-1, V + 1), tgt.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 300 == 0:
        print(f'step {step:4d}: loss={loss.item():.3f}')
print('\nTeacher-forced loss decreases - the decoder learns to copy with reversed order.')


## 11. Inference: Greedy Decoding (No Teacher)

At inference we feed in the model's *own* previous prediction, one token at a time.


In [ ]:
def greedy_decode(model, src, L=L):
    batch = src.size(0)
    tgt = torch.full((batch, 1), BOS)
    for _ in range(L):
        logits, _ = model(src, tgt, tgt_mask=causal_mask(tgt.size(1)))
        nxt = logits[:, -1].argmax(-1).unsqueeze(1)
        tgt = torch.cat([tgt, nxt], dim=1)
    return tgt[:, 1:]

model.eval()
with torch.no_grad():
    src_test, _, tgt_test = gen_batch(2000)
    preds = greedy_decode(model, src_test)
    acc = (preds == tgt_test).all(dim=1).float().mean().item()
print(f'Exact-sequence decoding accuracy: {acc*100:.0f}%')
i = 5
print(f'example src: {src_test[i].tolist()}')
print(f'example out:  {preds[i].tolist()}')
print(f'example tgt:  {tgt_test[i].tolist()}')


## 12. Failure Case: Missing Causal Mask in the Decoder

Without the causal mask the decoder attends to future gold tokens at position i -> trivially low loss during training that collapses at inference.


In [ ]:
with torch.no_grad():
    src, dec_in, tgt = gen_batch(8)
    # correct: causal mask
    l_masked, _ = model(src, dec_in, tgt_mask=causal_mask(L))
    # broken: no mask at all -> position i sees future positions
    l_unmasked, _ = model(src, dec_in, tgt_mask=None)

def top1_prob(logits, tgt):
    p = F.softmax(logits, dim=-1)
    return p.gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean().item()

print(f'P(gold token) with causal mask:  {top1_prob(l_masked, tgt):.3f}')
print(f'P(gold token) WITHOUT mask:      {top1_prob(l_unmasked, tgt):.3f}')
print('\nThe unmasked decoder peeks at the answer that follows -> inflated trust.')


## 13. Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Decoder repeats tokens | Mask too permissive | Check mask is lower-triangular | Apply tgt_mask |
| Training loss flat | Cross-attention disconnected | Perturb memory, watch logits | Wire memory into decoder |
| Train good / inference bad | Teacher forcing exposure bias | Compare forced vs free-run | Scheduled sampling |
| Output length fixed | No EOS handling | Check if model learns EOS | Add EOS, use beam search |

## 14. Real-World Considerations

- Share the embedding matrix between encoder and decoder when vocabularies match (fewer parameters).
- T5 and BART are production encoder-decoder transformers; GPT-style models are decoder-only and have no cross-attention.
- Teacher forcing optimizes next-token accuracy, not sequence accuracy - evaluate with beam search or sampling.

## 15. Common Mistakes

- Forgetting the causal mask (cheating).
- Bidirectional attention in the decoder (breaks autoregression).
- Confusing cross-attention (Q=decoder, K/V=encoder) with self-attention.
- Evaluating with teacher forcing instead of greedy decoding.

## 16. When NOT to Use

- Simple classification: encoder-only (BERT) is cheaper.
- Open-ended generation: decoder-only (GPT) is simpler.
- Input/output have the same structure: a decoder-only or encoder-only model may suffice.

## 17. Challenge: Scheduled Sampling

Instead of always feeding gold tokens, feed the model's own prediction with probability e. This should interpolate between teacher forcing and free run.


In [ ]:
def train_scheduled(model, steps=500, e_sched=(0.0, 0.3)):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    for step in range(steps):
        e = e_sched[0] + (e_sched[1] - e_sched[0]) * step / steps
        src, dec_in, tgt = gen_batch(64)
        # with prob e, replace a prefix of the decoder input with model predictions
        if e > 0:
            with torch.no_grad():
                pred_full = greedy_decode(model, src)
            r = torch.bernoulli(torch.full((1,), e)).item()
            if r == 1:
                dec_in = torch.cat([torch.full((len(src), 1), BOS), pred_full[:, :-1]], dim=1)
        opt.zero_grad()
        logits, _ = model(src, dec_in, tgt_mask=causal_mask(L))
        loss = loss_fn(logits.reshape(-1, V + 1), tgt.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    with torch.no_grad():
        src_test, _, tgt_test = gen_batch(2000)
        acc = (greedy_decode(model, src_test) == tgt_test).all(1).float().mean().item()
    return acc

# reuse the already-trained model and expose it to scheduled sampling
model.train()
acc_after = train_scheduled(model, steps=500, e_sched=(0.0, 0.35))
print(f'Decoding accuracy after scheduled-sampling: {acc_after*100:.0f}%')
print('Bridging teacher forcing and free run can harden generation against errors.')


## 18. Closed-Book Recall

1. In cross-attention, where do Q, K, V come from?
2. What is teacher forcing and what problem does it cause?
3. Why does the decoder need a causal mask but the encoder does not?
4. How do you generate a sequence at inference time?

## 19. Teach-Back Questions

Explain to another person:

- The encoder-reads / decoder-writes mental model with cross-attention.
- Why the no-mask decoder 'cheats' during training.
- The reverse-sequence task and how greedy decoding works.

## 20. Summary

You built a full encoder-decoder transformer, proved cross-attention really reads encoder memory, trained it to reverse sequences with teacher forcing, decoded greedily at inference, reproduced the classic missing-mask failure, and applied scheduled sampling.

## 21. Further Experiment

- Predict the hard task: train to REVERSE with length 12 and watch accuracy drop.
- Implement beam search (width 3) and compare exact-match accuracy vs greedy.
- Add EOS and train open-ended generation instead of fixed length.

## 22. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
